In [1]:
import pyreadstat 
import numpy as np
import pandas as pd



In [3]:
# read the .sav file
dataset, meta = pyreadstat.read_sav("../data/CMIR71FL.sav")

# see all variable labels
var_labels = pd.DataFrame({
    'variable': list(meta.column_names),
    'label': list(meta.column_labels)
})

def get_column_info(varname,value=False):
    if value:
        return meta.variable_value_labels.get(varname, 'not found')
    else:
        return meta.column_names_to_labels.get(varname, 'not found')

# Get the label for a specific variable
meta.column_names_to_labels.get('V632', 'not found')

# see all value labels for a specific variable
meta.variable_value_labels.get('V632', 'not found')

dataset.shape #(14677, 5102)




(14677, 5102)

In [ ]:
# Filter the dataset to include only the relevant columns
married_dataset = dataset[(dataset['V502'] == 1) & (dataset['V213'] == 0) & (dataset['V312']!=0)].copy()  # married/living with a man and not pregnant and using contraceptive


married_dataset["fp_decision_autonomy"] = married_dataset["V632"] 

# Applying the sampling weight to the dataset
married_dataset["weight"] = married_dataset["V005"] / 1000000 # V005 --- Sampling weight (divide by 1,000,000 to get the actual weight)

married_dataset.shape # (1533, 5104)



(1533, 5104)

In [ ]:

candidate_vars = {
    # LEVEL 1 — SOCIO-DEMOGRAPHIC FACTORS
    "V013":  "Age",
    "V025":  "Residence (1=Urban, 2=Rural)",
    "V130":  "Religion",
    "V024":  "Region",
    "V501": "Marital Status",
    "V190":  "Wealth index",
    


    # LEVEL 2 — INDIVIDUAL FACTORS
    "V714":  "Woman currently working (0=No, 1=Yes)",
    "V602": "Fertility preference",
    "V106":  "Woman's education level ",
    "V313":  "Current contraceptive method used ",
    "V218":  "Number of living children",
    
    # LEVEL 3 — INTERPERSONAL FACTORS
    "V505":  "number of other wives (0=monogamous, 1+=polygynous)",
    "V705": "Partner's occupation", # will be recoded
    "V701": "Husband/partner's education level", # will be recoded to know if he has a higher level than wife
    "V621": "Partner's desire for children",

    # LEVEL 4 — INSTITUTIONAL / HEALTH SERVICE FACTORS
    "M14$1":   "Number of ANC visits during last pregnancy",
    "V393A": "Fieldworker talked about FP",
    "V395":  "Health facility staff talked about FP",
    "V384A": "Heard FP message on radio (0=No, 1=Yes)",
    "V384B": "Heard FP message on TV (0=No, 1=Yes)",
    "V384C": "Heard FP message in newspaper/magazine (0=No, 1=Yes)",
    "V384D": "Heard FP message via mobile phone (0=No, 1=Yes)",
}

print(f"Total candidate variables: {len(candidate_vars)}")  # 22

# Inspect each variable
print("VARIABLE INSPECTION REPORT")

for var, desc in candidate_vars.items():
    print(f"VARIABLE: {var} | {desc}")
    
    if var not in married_dataset.columns:
        print(f"⚠️  WARNING: {var} NOT FOUND in dataset!")
        continue
    
    # Value counts
    vc = married_dataset[var].value_counts(dropna=False).sort_index()
    print("\nValue distribution:")
    print(vc.head(15))
    
    # Check for special DHS codes
    special_codes = [99, 98, 97, 96, 95, 94, 93, 88, 77]
    found_special = []
    for code in special_codes:
        if code in vc.index:
            found_special.append(f"{code} (n={vc[code]})")
    
    if found_special:
        print(f"\n⚠️  Special codes found: {', '.join(found_special)}")
    
    # Missing
    missing = married_dataset[var].isna().sum()
    if missing > 0:
        print(f"\nMissing: {missing} ({missing/len(married_dataset)*100:.1f}%)")




Total candidate variables: 22
VARIABLE INSPECTION REPORT
VARIABLE: V013 | Age

Value distribution:
V013
1.0     81
2.0    273
3.0    364
4.0    336
5.0    262
6.0    139
7.0     78
Name: count, dtype: int64
VARIABLE: V025 | Residence (1=Urban, 2=Rural)

Value distribution:
V025
1.0    911
2.0    622
Name: count, dtype: int64
VARIABLE: V130 | Religion

Value distribution:
V130
1.0     675
2.0     438
3.0     175
4.0     212
5.0       3
7.0      23
96.0      7
Name: count, dtype: int64

⚠️  Special codes found: 96 (n=7)
VARIABLE: V024 | Region

Value distribution:
V024
1.0      42
2.0     183
3.0     133
4.0     294
5.0      81
6.0     113
7.0      87
8.0      95
9.0     191
10.0    109
11.0     33
12.0    172
Name: count, dtype: int64
VARIABLE: V501 | Marital Status

Value distribution:
V501
1.0    976
2.0    557
Name: count, dtype: int64
VARIABLE: V190 | Wealth index

Value distribution:
V190
1.0     92
2.0    288
3.0    389
4.0    359
5.0    405
Name: count, dtype: int64
VARIABLE: V71

In [5]:
get_column_info("D109",True)
dataset["D109"].value_counts(dropna=False).sort_index()


D109
0.0       417
1.0       178
2.0       302
3.0       197
4.0        91
5.0       123
6.0        45
7.0        28
8.0        20
9.0        10
10.0       41
11.0        4
12.0       13
13.0        6
14.0        4
15.0       14
16.0        6
17.0        4
18.0        2
19.0        2
20.0        4
21.0        1
22.0       22
23.0        2
24.0        3
25.0        2
27.0        2
28.0        1
95.0      111
NaN     13022
Name: count, dtype: int64

In [ ]:

# COMPLETE RECODING FOR ALL  VARIABLES


print("STARTING RECODING PROCESS")
# --- LEVEL 1: SOCIO-DEMOGRAPHIC FACTORS ---

print("LEVEL 1: SOCIO-DEMOGRAPHIC")

# 1. AGE (V013) - Already in groups, but we'll also keep continuous V012
print("\n1. Age...")
married_dataset['age_group'] = married_dataset['V013']  # 1=15-19, 2=20-24, etc.
married_dataset['age'] = married_dataset['V012']  # Continuous age
print(f"   Age groups: {sorted(married_dataset['age_group'].dropna().unique())}")

# 2. RESIDENCE (V025) 
print("\n2. Residence...")
married_dataset['residence'] = married_dataset['V025']
print(f"   Urban: {(married_dataset['residence']==1).sum()}, Rural: {(married_dataset['residence']==2).sum()}")

# 3. WOMAN'S EDUCATION (V106)
print("\n3. Woman's education...")
married_dataset['edu_woman'] = married_dataset['V106']  # No missing values here
# 0=None, 1=Primary, 2=Secondary, 3=Higher
print(f" Recoded: {married_dataset['edu_woman'].value_counts().sort_index().to_dict()}")

# 4. RELIGION (V130)
print("\n6. Religion...")
# Recode 
married_dataset['religion'] = married_dataset['V130']
print(f"   Recoded: {married_dataset['religion'].value_counts().to_dict()}")

# 5. REGION (V024)
print("\n6. Region...")
# region_map = {
#     1.0: 1,
#     2.0: 2,
#     3.0: 3,
#     4.0: 4,
#     5.0: 5,
#     6.0: 3,
#     7.0: 6,
#     8.0: 7,
#     9.0: 8,
#     10.0: 9,
#     11.0: 10,
#     12.0: 2
# }
# married_dataset['region'] = married_dataset['V024'].map(region_map);
married_dataset['region'] = married_dataset['V024'];
print(f"  Recoded: {married_dataset['region'].value_counts().to_dict()}")

# 6. MARITAL STATUS 
print("\n6. MARITAL STATUS...")
marital_map = { 1.0: 1, #'Married',
 2.0: 2 #'Cohabitating',
}
married_dataset['marital_status'] = married_dataset['V501'].map(marital_map)
print(f"   Recoded: {married_dataset['marital_status'].value_counts().to_dict()}")

# 7. NUMBER OF LIVING CHILDREN (CATEGORICAL AND NUMERICAL)
print("\n7. NUMBER OF LIVING CHILDREN")
children_map = {}
for i in range(0,14):
    if i<3:
      children_map[i] = 1 #"<3"
    elif i>=3 and i<=5:
       children_map[i] = 2 # "3-5"
    elif i>5:
       children_map[i] = 3 # ">5"

married_dataset['num_children'] = married_dataset['V218']
married_dataset['children_group'] = married_dataset['V218'].map(children_map)
print(f"   Recoded: {married_dataset['children_group'].value_counts().to_dict()}")

# 8. WEALTH INDEX (V190)
print("\n8. Wealth index...")
married_dataset['wealth'] = married_dataset['V190']  
print(f"  Distribution: {married_dataset['wealth'].value_counts().sort_index().to_dict()}")

# 9. POLYGYNY (V505)
print("\n9. Type of marriage...")
def recode_type(v):
    if v==0:
        return 1 # "Monogamy"
    elif v==98:
        return 3 # "Don't Know"
    elif v>0:
        return 2 # "Polygamy"
    
married_dataset['marriage_type'] = married_dataset["V505"].apply(recode_type)  
print(f" Distribution: {married_dataset['marriage_type'].value_counts()}")


# --- LEVEL 2: INDIVIDUAL FACTORS ---

print("LEVEL 2: INDIVIDUAL FACTORS")

# 10. WOMAN WORKING (V714)
print("\n10. Woman currently working...")
married_dataset['woman_working'] = married_dataset['V714'].copy()
print(f"   Recoded: {married_dataset['woman_working'].value_counts()}")


# 12. FERTILITY PREFERENCES
print("\n12. Fertility Preferences")
print(f"   Original: {married_dataset['V602'].value_counts().sort_index().to_dict()}")
def recode_fertility(v):
    if v == 1:
        return 1  # "Wants More"
    elif v == 2 or v== 3 or v == 6:
        return 2  # "Doesn't want more/ Undecided"
    elif v == 4 or v== 5:
        return 3  # "Cannot have children"
    else:
        return np.nan
    
married_dataset['fertility_preference'] = married_dataset['V602'].apply(recode_fertility)


# 14. CURRENT CONTRACEPTIVE METHOD (V313)
print("\n14. Current contraceptive method...")
print(f"   Original: {married_dataset['V312'].value_counts().sort_index().to_dict()}")
married_dataset['current_method'] = married_dataset['V312']


# --- LEVEL 3: INTERPERSONAL FACTORS ---

print("LEVEL 3: INTERPERSONAL FACTORS")

# 15. HUSBAND'S EDUCATION (V701)
print("\n15. Husband's education...")
married_dataset['edu_husband'] = married_dataset['V701'] # we will treat "Dont Know" as a category
print(f"   Recoded: {married_dataset['edu_husband'].value_counts().sort_index().to_dict()}")

# 16. HUSBAND'S OCCUPATION (V705)
print("\n16. Husband's occupation...")
def recode_husb(v):
    if v==0:
        return 0
    elif v==98:
        return 2 # dont know
    else:
        return 1
    
married_dataset['husband_working'] = married_dataset['V705'].apply(recode_husb)
print(f"   Recoded: {married_dataset['husband_working'].value_counts().to_dict()}")


# 18. HUSBAND'S DESIRED CHILDREN (V621)
print("\n18. Husband's desired children...")
married_dataset['husband_desired_children'] = married_dataset['V621'].replace({ 99: np.nan})
print(f"   After cleaning - Range: {married_dataset['husband_desired_children'].min()} to {married_dataset['husband_desired_children'].max()}")

# --- LEVEL 4: INSTITUTIONAL/HEALTH SERVICE FACTORS ---

print("LEVEL 4: INSTITUTIONAL/HEALTH SERVICE")

# 19. ANC VISITS (M14$1) 
anc_var = 'M14$1' if 'M14$1' in married_dataset.columns else 'M141'
print(f"\n19. ANC visits (using {anc_var})...")
if anc_var in married_dataset.columns:
    
    married_dataset['anc_visits'] = married_dataset[anc_var].replace({98: np.nan, 99: np.nan})
    # Create groups
    married_dataset['anc_group'] = pd.cut(married_dataset['anc_visits'], 
                              bins=[-1, 0, 3, 100], 
                            
                              labels=[1, 2, 3])
    married_dataset['anc_group'] = married_dataset['anc_group'].cat.add_categories(0).fillna(0)
    # l                           labels=['0', '1-3', '3+'])
    print(f"   Groups: {married_dataset['anc_group'].value_counts().to_dict()}")
else:
    print("   ⚠️ Variable not found!")

# 20. FIELDWORKER TALKED ABOUT FP (V393A)
print("\n20. Fieldworker talked about FP...")
married_dataset['fieldworker_fp'] = married_dataset['V393A'].fillna(2)  # 1=Yes, 0=No, 2=dont know
 


# 21. HEALTH FACILITY STAFF TALKED ABOUT FP (V395)
print("\n21. Health facility staff talked about FP...")
married_dataset['facility_fp'] = married_dataset['V395'].fillna(2)  # 1=Yes, 0=No, 2=dont know


# 22-25. MEDIA EXPOSURE (Radio, TV, Newspaper, Mobile)
print("\n22-25. Media exposure...")
media_vars = ['V384A', 'V384B', 'V384C', 'V384D']
media_names = ['radio', 'tv', 'newspaper', 'mobile']

for var, name in zip(media_vars, media_names):
    if var in married_dataset.columns:
        print(f"\n   {var} ({name}):")
        print(f"   Original: {married_dataset[var].value_counts().sort_index().to_dict()}")
        married_dataset[f'media_{name}'] = married_dataset[var].replace({8: np.nan, 9: np.nan})
        print(f"   Heard FP on {name}: {(married_dataset[f'media_{name}']==1).sum()}")
    else:
        print(f"   ⚠️ {var} not found!")

# Create composite: ANY media exposure
media_cols = [c for c in married_dataset.columns if c.startswith('media_')]
married_dataset['media_any'] = (married_dataset[media_cols].sum(axis=1) > 0).astype(int)
print(f"\n   Any media exposure: {married_dataset['media_any'].sum()}")


print("RECODING COMPLETE!")


STARTING RECODING PROCESS
LEVEL 1: SOCIO-DEMOGRAPHIC

1. Age...
   Age groups: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]

2. Residence...
   Urban: 911, Rural: 622

3. Woman's education...
 Recoded: {0.0: 86, 1.0: 534, 2.0: 775, 3.0: 138}

6. Religion...
   Recoded: {1.0: 675, 2.0: 438, 4.0: 212, 3.0: 175, 7.0: 23, 96.0: 7, 5.0: 3}

6. Region...
  Recoded: {4.0: 294, 9.0: 191, 2.0: 183, 12.0: 172, 3.0: 133, 6.0: 113, 10.0: 109, 8.0: 95, 7.0: 87, 5.0: 81, 1.0: 42, 11.0: 33}

6. MARITAL STATUS...
   Recoded: {1: 976, 2: 557}

7. NUMBER OF LIVING CHILDREN
   Recoded: {2: 748, 1: 538, 3: 247}

8. Wealth index...
  Distribution: {1.0: 92, 2.0: 288, 3.0: 389, 4.0: 359, 5.0: 405}

9. Type of marriage...
 Distribution: marriage_type
1    1296
2     176
3      61
Name: count, dtype: int64
LEVEL 2: INDIVIDUAL FACTORS

10. Woman currently working...
   Recoded: woman_working
1.0    1152
0.0     381
Name: count, dtype: int64

12. Fertility Preferences
   Original: {1.0: 875, 2.0: 69, 3.0: 543, 4.0: 23, 

In [ ]:

# CREATE CLEAN ANALYSIS DATASET

# All recoded variables
recoded_vars = [
    # IDs and weights
    'caseid', 'cluster_id', 'weight',
    
    # Outcome 
    'fp_decision_autonomy',
    
    # Level 1: Socio-demographic
    "age_group", "age", "residence" ,"edu_woman","religion","region","marital_status",
    "num_children","children_group","wealth","marriage_type",
    
    # Level 2: Individual
    "woman_working","fertility_preference","current_method",
    
    # Level 3: Interpersonal
    "edu_husband","husband_working","husband_desired_children",
    
    # Level 4: Institutional
    "anc_visits","anc_group","fieldworker_fp","facility_fp","media_any"
]

# Filter to existing columns
existing_vars = [v for v in recoded_vars if v in married_dataset.columns]
analysis_married_dataset = married_dataset[existing_vars].copy()

# Add cluster ID for multilevel
analysis_married_dataset['cluster_id'] = married_dataset['V021']

print(f"\nFinal dataset shape: {analysis_married_dataset.shape}")
print(f"Variables: {len(analysis_married_dataset.columns)}")
print("\nVariables in dataset:")
for i, col in enumerate(analysis_married_dataset.columns, 1):
    missing = analysis_married_dataset[col].isna().sum()
    print(f"{i:2d}. {col:25} | Missing: {missing:4d} ({missing/len(analysis_married_dataset)*100:5.1f}%)")

# Save
analysis_married_dataset.to_csv("outputs/recoded_analysis_data.csv", index=False)
print(f"\n✓ Saved: recoded_analysis_data.csv")


Final dataset shape: (1533, 25)
Variables: 25

Variables in dataset:
 1. weight                    | Missing:    0 (  0.0%)
 2. fp_decision_autonomy      | Missing:    0 (  0.0%)
 3. age_group                 | Missing:    0 (  0.0%)
 4. age                       | Missing:    0 (  0.0%)
 5. residence                 | Missing:    0 (  0.0%)
 6. edu_woman                 | Missing:    0 (  0.0%)
 7. religion                  | Missing:    0 (  0.0%)
 8. region                    | Missing:    0 (  0.0%)
 9. marital_status            | Missing:    0 (  0.0%)
10. num_children              | Missing:    0 (  0.0%)
11. children_group            | Missing:    0 (  0.0%)
12. wealth                    | Missing:    0 (  0.0%)
13. marriage_type             | Missing:    0 (  0.0%)
14. woman_working             | Missing:    0 (  0.0%)
15. fertility_preference      | Missing:    0 (  0.0%)
16. current_method            | Missing:    0 (  0.0%)
17. edu_husband               | Missing:    0 (  0